Zadanie 2. 

Wybierz jeden z plików csv z poprzednich ćwiczeń (Mini Kurs, Actors.csv, Names.csv ect..) lub inny plik z dbfs:/databricks-datasets/ i na jego podstawie stwórz schemat danych. Przykład na wykładzie.  

Użyj funckji spark.read. i stwórz DataFrame (DataFrameReader) wczytując plik z użyciem schematu, który stworzyłeś. 

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/COVID/covid-19-data/"))


path,name,size,modificationTime
dbfs:/databricks-datasets/COVID/covid-19-data/.git/,.git/,0,0
dbfs:/databricks-datasets/COVID/covid-19-data/.github/,.github/,0,0
dbfs:/databricks-datasets/COVID/covid-19-data/.gitignore,.gitignore,10,1615898767000
dbfs:/databricks-datasets/COVID/covid-19-data/LICENSE,LICENSE,1289,1615898767000
dbfs:/databricks-datasets/COVID/covid-19-data/NEW-YORK-DEATHS-METHODOLOGY.md,NEW-YORK-DEATHS-METHODOLOGY.md,2771,1615898767000
dbfs:/databricks-datasets/COVID/covid-19-data/NYT-readme.md,NYT-readme.md,1748,1586273566000
dbfs:/databricks-datasets/COVID/covid-19-data/PROBABLE-CASES-NOTE.md,PROBABLE-CASES-NOTE.md,3162,1615898767000
dbfs:/databricks-datasets/COVID/covid-19-data/README.md,README.md,22959,1615898767000
dbfs:/databricks-datasets/COVID/covid-19-data/colleges/,colleges/,0,0
dbfs:/databricks-datasets/COVID/covid-19-data/excess-deaths/,excess-deaths/,0,0


In [0]:
#to see how does this look like
dbutils.fs.head("dbfs:/databricks-datasets/COVID/covid-19-data/us-counties.csv", 50)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

schema = StructType([
    StructField("date", StringType(), True),
    StructField("county", StringType(), True),
    StructField("state", StringType(), True),
    StructField("fips", IntegerType(), True),
    StructField("cases", IntegerType(), True),
    StructField("deaths", IntegerType(), True)
])


path = "dbfs:/databricks-datasets/COVID/covid-19-data/us-counties.csv"

df = spark.read.format("csv").option("header", "true").option("delimiter", ",").option("quote", '"').schema(schema).load(path)


df.printSchema() 
df.show(10)

root
 |-- date: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- fips: integer (nullable = true)
 |-- cases: integer (nullable = true)
 |-- deaths: integer (nullable = true)

+----------+-----------+----------+-----+-----+------+
|      date|     county|     state| fips|cases|deaths|
+----------+-----------+----------+-----+-----+------+
|2020-01-21|  Snohomish|Washington|53061|    1|     0|
|2020-01-22|  Snohomish|Washington|53061|    1|     0|
|2020-01-23|  Snohomish|Washington|53061|    1|     0|
|2020-01-24|       Cook|  Illinois|17031|    1|     0|
|2020-01-24|  Snohomish|Washington|53061|    1|     0|
|2020-01-25|     Orange|California| 6059|    1|     0|
|2020-01-25|       Cook|  Illinois|17031|    1|     0|
|2020-01-25|  Snohomish|Washington|53061|    1|     0|
|2020-01-26|   Maricopa|   Arizona| 4013|    1|     0|
|2020-01-26|Los Angeles|California| 6037|    1|     0|
+----------+-----------+----------+-----+-----+------+

Zadanie 3 

Użycie Read Modes.  

Wykorzystaj posiadane pliki bądź użyj nowe.  Użyj Sparka do odczytania jednego pliku i użyj wszystkich typów read modes. Poprawny plik nie wywoła żadnych efektów, więc popsuj dane tak aby każda z read modes zadziałał. 

In [0]:
from pyspark.sql.functions import when, col
df_corrupted = df.withColumn("fips",when(col("county") == "Snohomish", "flips").otherwise(col("fips"))).withColumn("deaths", when(col("state") == "Illinois", "NaN").otherwise(col("deaths")))

display(df_corrupted.limit(10))
df_corrupted.limit(10).write.format("csv").mode("overwrite").option("header", "true").save("dbfs:/FileStore/tables/us-counties-corrupted.csv")

filePath2 = "dbfs:/FileStore/tables/us-counties-corrupted.csv"

date,county,state,fips,cases,deaths
2020-01-21,Snohomish,Washington,flips,1,0
2020-01-22,Snohomish,Washington,flips,1,0
2020-01-23,Snohomish,Washington,flips,1,0
2020-01-24,Cook,Illinois,17031,1,NaN
2020-01-24,Snohomish,Washington,flips,1,0
2020-01-25,Orange,California,6059,1,0
2020-01-25,Cook,Illinois,17031,1,NaN
2020-01-25,Snohomish,Washington,flips,1,0
2020-01-26,Maricopa,Arizona,4013,1,0
2020-01-26,Los Angeles,California,6037,1,0


In [0]:
#PERMISSIVE - Inserts null for corrupted rows
df_permissive = spark.read.format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load(filePath2)
display(df_permissive.limit(10))

#DROPMALFORMED - Skips malformed rows
df_dropmalformed = spark.read.format("csv") \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(schema) \
    .load(filePath2)

display(df_dropmalformed.limit(10))

#FAILFAST - Stops reading at the first error
df_failfast = spark.read.format("csv") \
    .option("header", "true") \
    .option("mode", "FAILFAST") \
    .schema(schema) \
    .load(filePath2)

display(df_failfast.limit(10))

date,county,state,fips,cases,deaths
2020-01-21,Snohomish,Washington,null,1,0
2020-01-22,Snohomish,Washington,null,1,0
2020-01-23,Snohomish,Washington,null,1,0
2020-01-24,Cook,Illinois,17031,1,null
2020-01-24,Snohomish,Washington,null,1,0
2020-01-25,Orange,California,6059,1,0
2020-01-25,Cook,Illinois,17031,1,null
2020-01-25,Snohomish,Washington,null,1,0
2020-01-26,Maricopa,Arizona,4013,1,0
2020-01-26,Los Angeles,California,6037,1,0


date,county,state,fips,cases,deaths
2020-01-25,Orange,California,6059,1,0
2020-01-26,Maricopa,Arizona,4013,1,0
2020-01-26,Los Angeles,California,6037,1,0


org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 69.0 failed 1 times, most recent failure: Lost task 0.0 in stage 69.0 (TID 536) (ip-10-172-228-2.us-west-2.compute.internal executor driver): com.databricks.sql.io.FileReadException: Error while reading file dbfs:/FileStore/tables/us-counties-corrupted.csv/part-00000-tid-4439091527033067830-057e058a-7eb9-414f-b3ef-9f370bb4d12f-533-1-c000.csv.
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1$$anon$2.logFileNameAndThrow(FileScanRDD.scala:704)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1$$anon$2.getNext(FileScanRDD.scala:673)
	at org.apache.spark.util.NextIterator.hasNext(NextIterator.scala:73)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:796)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.$anonfun$hasNext$1(FileScanRDD.scala:496)
	at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.java:23)
	at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:486)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.collect.UnsafeRowBatchUtils$.encodeUnsafeRows(UnsafeRowBatchUtils.scala:82)
	at org.apache.spark.sql.execution.collect.Collector.$anonfun$processFunc$1(Collector.scala:208)
	at org.apache.spark.scheduler.ResultTask.$anonfun$runTask$3(ResultTask.scala:75)
	at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110)
	at org.apache.spark.scheduler.ResultTask.$anonfun$runTask$1(ResultTask.scala:75)
	at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:55)
	at org.apache.spark.scheduler.Task.doRunTask(Task.scala:179)
	at org.apache.spark.scheduler.Task.$anonfun$run$5(Task.scala:142)
	at com.databricks.unity.EmptyHandle$.runWithAndClose(UCSHandle.scala:126)
	at org.apache.spark.scheduler.Task.$anonfun$run$1(Task.scala:142)
	at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110)
	at org.apache.spark.scheduler.Task.run(Task.scala:97)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$13(Executor.scala:904)
	at org.apache.spark.util.Utils$.tryWithSafeFinally(Utils.scala:1741)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$4(Executor.scala:907)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110)
	at org.apache.spark.executor.Executor$TaskRunner.run(Executor.scala:761)
	at java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1149)
	at java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:624)
	at java.lang.Thread.run(Thread.java:750)
Caused by: org.apache.spark.SparkException: Malformed records are detected in record parsing. Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.malformedRecordsDetectedInRecordParsingError(QueryExecutionErrors.scala:1936)
	at org.apache.spark.sql.catalyst.util.FailureSafeParser.parse(FailureSafeParser.scala:103)
	at org.apache.spark.sql.catalyst.csv.UnivocityParser$.$anonfun$parseIterator$2(UnivocityParser.scala:507)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at org.apache.spark.util.CompletionIterator.hasNext(CompletionIterator.scala:31)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1$$anon$2.getNext(FileScanRDD.scala:619)
	... 29 more
Caused by: org.apache.spark.sql.catalyst.util.BadRe

Zadanie 4 

Użycie DataFrameWriter. 

Zapisz jeden z wybranych plików do formatów (‘.parquet’, ‘.json’). Sprawdź, czy dane są zapisane poprawnie, użyj do tego spark.read (DataFrameReader).  

In [0]:

df_corrupted.write.format("parquet").mode("overwrite").save("dbfs:/FileStore/tables/us-counties-corrupted.parquet")
df_corrupted.write.format("json").mode("overwrite").save("dbfs:/FileStore/tables/us-counties-corrupted.json")

df_parquet = spark.read.format("parquet").load("dbfs:/FileStore/tables/us-counties-corrupted.parquet")
df_json = spark.read.format("json").load("dbfs:/FileStore/tables/us-counties-corrupted.json")

display(df_json)
display(df_parquet)


cases,county,date,deaths,fips,state
1,Snohomish,2020-01-21,0,flips,Washington
1,Snohomish,2020-01-22,0,flips,Washington
1,Snohomish,2020-01-23,0,flips,Washington
1,Cook,2020-01-24,NaN,17031,Illinois
1,Snohomish,2020-01-24,0,flips,Washington
1,Orange,2020-01-25,0,6059,California
1,Cook,2020-01-25,NaN,17031,Illinois
1,Snohomish,2020-01-25,0,flips,Washington
1,Maricopa,2020-01-26,0,4013,Arizona
1,Los Angeles,2020-01-26,0,6037,California


date,county,state,fips,cases,deaths
2020-10-10,Tucker,West Virginia,54093,46,0
2020-10-10,Tyler,West Virginia,54095,20,0
2020-10-10,Upshur,West Virginia,54097,175,0
2020-10-10,Wayne,West Virginia,54099,437,11
2020-10-10,Webster,West Virginia,54101,9,0
2020-10-10,Wetzel,West Virginia,54103,68,0
2020-10-10,Wirt,West Virginia,54105,20,0
2020-10-10,Wood,West Virginia,54107,408,6
2020-10-10,Wyoming,West Virginia,54109,136,5
2020-10-10,Adams,Wisconsin,55001,316,4
